# LSA and LSI: Where Document Vectors Came From

**Computational Text Analysis Pack · Module 06 — Clustering Documents**

*Technique:* Truncated SVD on a term-document matrix; the ancestor of embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/05-unsupervised-structure/06d_lsa_and_the_vector_lineage.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Notebook 06a built a TF-IDF matrix: thousands of columns, one per word, almost
all of them zero. **Latent Semantic Analysis** (Deerwester et al., 1990) asks
whether that matrix has a much smaller number of real dimensions hiding inside
it, and finds them with the singular value decomposition.

This matters for two reasons. It fixes a concrete problem — two documents about
the same thing in different words look unrelated to TF-IDF — and it is the
direct ancestor of the embedding methods in module 23. LSA is where "a document
is a point in a few hundred dimensions" begins.

## The tooling

`sklearn.decomposition.TruncatedSVD` does LSA on a sparse matrix (plain `PCA`
would need to densify it and blow up your memory). **gensim** offers `LsiModel`
for the same job at corpus scale, and is the usual choice once the matrix stops
fitting in RAM.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

sub = df[(df["language"] == "English") & (df["n_words"] >= 30)].reset_index(drop=True)
vec = TfidfVectorizer(token_pattern=r"\w+", min_df=3, max_features=6000,
                      sublinear_tf=True)
X = vec.fit_transform(sub["text"])
vocab = np.array(vec.get_feature_names_out())
print(f"term-document matrix: {X.shape[0]} docs x {X.shape[1]} terms")
print(f"sparsity: {1 - X.nnz / (X.shape[0] * X.shape[1]):.3%} of cells are zero")

## Decomposing

SVD factors the matrix into components ordered by how much variance each
explains. Keeping the first *k* is the whole method: **k is the only real
parameter, and it is a modelling decision, not a default.**

In [ ]:
K = 100                     # <- change this
svd = TruncatedSVD(n_components=K, random_state=0)
D = svd.fit_transform(X)
print(f"reduced: {X.shape[1]} dimensions -> {K}")
print(f"variance retained: {svd.explained_variance_ratio_.sum():.1%}")
print()
cum = np.cumsum(svd.explained_variance_ratio_)
for k in [5, 10, 25, 50, 100]:
    if k <= K:
        print(f"   first {k:>3} components explain {cum[k-1]:.1%}")

plt.figure(figsize=(6.5, 3.6))
plt.plot(range(1, K + 1), cum)
plt.xlabel("components kept"); plt.ylabel("cumulative variance explained")
plt.title("How many dimensions does this corpus really have?")
plt.grid(alpha=.3); plt.tight_layout()

## What a component is

Each component is a weighted combination of words. Read the extremes of a few
and they are usually interpretable — which is LSA's advantage over the neural
methods that replaced it, and why it survives in DH work that has to be argued
rather than merely deployed.

In [ ]:
for i in range(4):
    order = np.argsort(svd.components_[i])
    print(f"component {i}")
    print("   + " + ", ".join(vocab[order[::-1]][:9]))
    print("   - " + ", ".join(vocab[order][:9]))
    print()

In [ ]:
# The problem LSA was invented to fix: vocabulary mismatch.
from sklearn.metrics.pairwise import cosine_similarity

probe = ["the president addressed the nation about the economy",
         "the head of state spoke to citizens regarding financial matters",
         "the football team lost the match on saturday"]
Ptf = vec.transform(probe)
Plsa = svd.transform(Ptf)
print("TF-IDF cosine (shares almost no words):")
print(np.round(cosine_similarity(Ptf), 3))
print("\nLSA cosine (same k as above):")
print(np.round(cosine_similarity(Plsa), 3))
print()
print("Sentences 1 and 2 mean nearly the same thing in different words. TF-IDF")
print("cannot see it; LSA moves them closer. That is the whole pitch, and it is")
print("the same pitch module 23's embeddings make with far more machinery.")

### 🔧 Your turn 1

Set K to 5 and then to 300, and re-run the probe. Where does the similarity between sentences 1 and 2 peak?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Very low k over-merges — everything looks similar because you have thrown away
the distinctions along with the noise. Very high k approaches the original
TF-IDF and the benefit disappears. The useful range is usually in the low
hundreds, and the honest way to choose is to plot the variance curve above and
pick where it flattens, then report the number. LSA has exactly one knob and
papers still routinely fail to say what they set it to.

</details>